# Tarea Jueves 6 de Agosto
## Ejercicios Prácticos: Método de Montecarlo en Python

El método de Montecarlo consiste en usar números aleatorios para aproximar resultados matemáticos que serían difíciles (o imposibles) de calcular de forma exacta. A continuación se resuelven los tres ejercicios de la tarea: uno básico, uno medio y uno avanzado.

## 1. Nivel Básico: Estimación de π

**Problema:** Implementar un programa que aproxime el valor de la constante matemática π simulando el lanzamiento de dardos aleatorios sobre un cuadrado de lado 2 que inscribe un círculo de radio 1.

**Fundamento matemático:**

El área del círculo es $A_c = \pi r^2 = \pi$, y el área del cuadrado es $A_q = (2r)^2 = 4$. La probabilidad de que un punto aleatorio $(x, y)$ distribuido uniformemente en el cuadrado caiga dentro del círculo es igual a la proporción de las áreas:

$$P(x^2 + y^2 \le 1) = \frac{\text{Área del círculo}}{\text{Área del cuadrado}} = \frac{\pi}{4}$$

Es decir, si lanzamos muchos "dardos" al azar dentro del cuadrado, la proporción que cae dentro del círculo, multiplicada por 4, nos da una aproximación de π.

In [7]:
import random
import math

def estimar_pi(n_puntos):
    """Aproxima el valor de pi lanzando 'dardos' aleatorios sobre un cuadrado."""
    puntos_dentro_circulo = 0

    for _ in range(n_puntos):
        # Generamos un punto aleatorio (x, y) dentro del cuadrado [-1, 1] x [-1, 1]
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)

        # Si la distancia al origen es menor o igual a 1, el punto cae dentro del círculo
        if x**2 + y**2 <= 1:
            puntos_dentro_circulo += 1

    # La proporción de puntos dentro del círculo, multiplicada por 4, aproxima a pi
    pi_estimado = 4 * (puntos_dentro_circulo / n_puntos)
    return pi_estimado


n = 100000
resultado = estimar_pi(n)

print(f"Número de puntos simulados: {n}")
print(f"Valor estimado de pi: {resultado}")
print(f"Valor real de pi:     {math.pi}")

Número de puntos simulados: 100000
Valor estimado de pi: 3.1464
Valor real de pi:     3.141592653589793


## 2. Nivel Medio: Integración Numérica

**Problema:** Utilizar el método de Montecarlo para calcular la integral definida de la función $f(x) = e^{-x^2}$ en el intervalo $[0, 2]$.

**Fundamento matemático:**

La aproximación de una integral definida mediante el método de Montecarlo se basa en calcular el valor esperado de la función sobre el intervalo de integración. Si tomamos variables aleatorias uniformemente distribuidas $X_i \sim U(a, b)$, la integral se aproxima como:

$$\int_a^b f(x)\,dx \approx \frac{b-a}{N}\sum_{i=1}^{N} f(x_i)$$

Es decir: promediamos el valor de la función evaluada en muchos puntos aleatorios del intervalo, y multiplicamos ese promedio por el ancho del intervalo $(b - a)$.

In [8]:
import random
import math

def f(x):
    """Función a integrar: f(x) = e^(-x^2)"""
    return math.exp(-x**2)


def integrar_montecarlo(funcion, a, b, n_puntos):
    """Aproxima la integral de 'funcion' en el intervalo [a, b]."""
    suma_evaluaciones = 0

    for _ in range(n_puntos):
        # Generamos un punto aleatorio dentro del intervalo [a, b]
        x_aleatorio = random.uniform(a, b)
        suma_evaluaciones += funcion(x_aleatorio)

    # Promedio de la función multiplicado por el ancho del intervalo
    integral_estimada = (b - a) * (suma_evaluaciones / n_puntos)
    return integral_estimada


n = 100000
a, b = 0, 2
resultado = integrar_montecarlo(f, a, b, n)

print(f"Número de puntos simulados: {n}")
print(f"Integral estimada de f(x) = e^(-x^2) en [{a}, {b}]: {resultado}")

Número de puntos simulados: 100000
Integral estimada de f(x) = e^(-x^2) en [0, 2]: 0.8822446662689184


## 3. Nivel Avanzado: Valuación de Opciones Financieras

**Problema:** Estimar el precio de una Opción de Compra Europea (European Call Option) simulando múltiples trayectorias del precio de una acción mediante el modelo estocástico de Movimiento Browniano Geométrico.

**Fundamento matemático:**

Bajo el modelo de Black-Scholes, el precio de la acción en el vencimiento $T$ sigue la ecuación:

$$S_T = S_0 \exp\left(\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z\right)$$

donde $Z \sim \mathcal{N}(0, 1)$ es una variable aleatoria normal estándar. El precio de la opción de compra $C$ es el valor presente del pago esperado al vencimiento:

$$C = e^{-rT}\,\mathbb{E}[\max(S_T - K, 0)]$$

Donde:
- $S_0$: precio inicial de la acción
- $K$: precio de ejercicio (strike)
- $r$: tasa libre de riesgo
- $\sigma$: volatilidad de la acción
- $T$: tiempo hasta el vencimiento (en años)

In [9]:
import random
import math

def precio_opcion_call(S0, K, r, sigma, T, n_simulaciones):
    """Estima el precio de una opción call europea con el método de Montecarlo."""
    suma_pagos = 0

    for _ in range(n_simulaciones):
        # Generamos un número aleatorio de una distribución normal estándar
        z = random.gauss(0, 1)

        # Calculamos el precio simulado de la acción al vencimiento (ecuación 3)
        ST = S0 * math.exp((r - (sigma**2) / 2) * T + sigma * math.sqrt(T) * z)

        # El pago de la opción es max(ST - K, 0)
        pago = max(ST - K, 0)
        suma_pagos += pago

    # Promediamos los pagos y los traemos a valor presente (ecuación 4)
    pago_esperado = suma_pagos / n_simulaciones
    precio_opcion = math.exp(-r * T) * pago_esperado
    return precio_opcion


# Parámetros de ejemplo
S0 = 100      # Precio inicial de la acción
K = 105       # Precio de ejercicio (strike)
r = 0.05      # Tasa libre de riesgo (5% anual)
sigma = 0.2   # Volatilidad (20% anual)
T = 1         # Tiempo hasta el vencimiento (1 año)
n = 100000    # Número de simulaciones

resultado = precio_opcion_call(S0, K, r, sigma, T, n)

print(f"Número de simulaciones: {n}")
print(f"Precio estimado de la opción call europea: {resultado:.4f}")

Número de simulaciones: 100000
Precio estimado de la opción call europea: 8.0942
